# Protein Sequence Mapping

This notebook maps protein IDs from your metabolic model to NCBI protein accessions using BLASTp.
The output CSV is used in **Step 2** of the Universal Metabolic Community Pipeline.

---

### Instructions
1. Click **Runtime → Run all** from the top menu
2. Upload your files when prompted
3. Wait for BLAST to finish — the CSV will appear in the Files panel on the left (📁)
4. Right-click the CSV and select **Download**
5. Upload it to Step 2 of the pipeline

In [ ]:
#@title Step 1: Install BLAST { display-mode: "form" }
#@markdown Click the run button (▶) on the left to install BLAST. This takes about 1 minute.
import shutil, subprocess
if not shutil.which('blastp'):
    print('Installing BLAST...')
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ncbi-blast+'], check=True)
    print('✅ BLAST installed.')
else:
    print('✅ BLAST already installed.')
import subprocess
subprocess.run(['pip', 'install', '-q', 'pandas'], capture_output=True)

In [ ]:
#@title Step 2: Upload your FASTA files { display-mode: "form" }
#@markdown Upload your **query FASTA** (model protein sequences) and **database FASTA** (NCBI protein sequences for the same species).
from google.colab import files
import os

print('📂 Upload your QUERY FASTA (.faa) — protein sequences from your metabolic model')
uploaded_query = files.upload()
query_filename_raw = list(uploaded_query.keys())[0]
query_filename = query_filename_raw.replace(' ', '_')
if query_filename != query_filename_raw:
    os.rename(query_filename_raw, query_filename)
print(f'✅ Query file: {query_filename}')

print('\n📂 Upload your DATABASE FASTA (.faa) — protein sequences downloaded from NCBI')
uploaded_db = files.upload()
db_filename_raw = list(uploaded_db.keys())[0]
db_filename = db_filename_raw.replace(' ', '_')
if db_filename != db_filename_raw:
    os.rename(db_filename_raw, db_filename)
print(f'✅ Database file: {db_filename}')

species_name = query_filename.split('_')[0]
print(f'\n🔬 Species detected: {species_name}')

In [ ]:
#@title Step 3: Run BLAST and generate mapping CSV { display-mode: "form" }
#@markdown This may take several minutes. When done, your CSV will appear in the Files panel on the left (📁).
import subprocess, pandas as pd, os

db_name     = f'{species_name}_db'
blast_out   = f'{species_name}_blast.txt'
output_csv  = f'{species_name}_protein_id_mapping.csv'

print('⚙️  Building BLAST database...')
subprocess.run(['makeblastdb', '-in', db_filename, '-dbtype', 'prot', '-out', db_name],
               check=True, capture_output=True)

print('⚙️  Running BLASTp...')
subprocess.run([
    'blastp', '-query', query_filename, '-db', db_name, '-out', blast_out,
    '-outfmt', '6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore qcovs',
    '-num_threads', '2'
], check=True, capture_output=True)

cols = ['qseqid','sseqid','pident','length','mismatch','gapopen','qstart','qend','sstart','send','evalue','bitscore','qcovs']
blast_df = pd.read_csv(blast_out, sep='\t', header=None, names=cols)

def seq_lengths(fasta):
    lengths, cur_id, cur_len = {}, None, 0
    with open(fasta) as f:
        for line in f:
            if line.startswith('>'):
                if cur_id: lengths[cur_id] = cur_len
                cur_id, cur_len = line[1:].strip().split()[0], 0
            else:
                cur_len += len(line.strip())
    if cur_id: lengths[cur_id] = cur_len
    return lengths

blast_df['qlen'] = blast_df['qseqid'].map(seq_lengths(query_filename))
blast_df['slen'] = blast_df['sseqid'].map(seq_lengths(db_filename))
blast_df['qcov'] = (blast_df['qend'] - blast_df['qstart'] + 1) / blast_df['qlen']
blast_df['scov'] = blast_df['length'] / blast_df['slen']

filtered = blast_df[
    (blast_df['pident']   >= 95) &
    (blast_df['gapopen']  == 0)  &
    (blast_df['mismatch'] <= 1)  &
    (blast_df['qcov']     >= 0.85) &
    (blast_df['scov']     >= 0.85)
]

best = filtered.sort_values('evalue').groupby('qseqid').first().reset_index()
result = best[['qseqid','sseqid']].rename(columns={'qseqid':'identifier_query','sseqid':'identifier_db'})
result.to_csv(output_csv, index=False)

print(f'\n✅ Done! {len(result)} matches found out of {blast_df["qseqid"].nunique()} sequences.')
print(f'📄 Output: {output_csv}')
print('\n👉 Find the file in the Files panel on the left (📁), right-click → Download.')